In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn import clone
from sklearn.model_selection import KFold
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['target'] = housing.target

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 1],
    'model__colsample_bytree': [0.7, 0.8, 1]
}

rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [6, 8, 12],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

models = [
    ('Random Forest', RandomForestRegressor(), rf_param_grid),
    ('XGBoost', XGBRegressor(), xgb_param_grid),
    ('Linear Regression', LinearRegression(), None),
    ('SVM Regressor', SVR(), None),
    ('KNN', KNeighborsRegressor(), None)
]

# We can also do, which is a standard way:
# models = {
# "Model Name": (model, param_grid),
# "Model Name": .....
# }

numeric_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
])

for name, model, param_grid in models:
    pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model)
    ])

    if param_grid:
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        y_pred = grid.predict(X_test)
        
        # Print the performance metrics
        print("Model:", name)
        print("Cross-validation R2:", grid.best_score_)
        print("R2 Score: ", r2_score(y_test, y_pred))
        print()

    else:
        scores = cross_val_score(pipeline, X_train, y_train, cv=5)
        mean_r2 = scores.mean()
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        # Print the performance metrics
        print("Model:", name)
        print("Cross-validation Accuracy:", mean_r2)
        print("R2 Score: ", r2_score(y_test, y_pred))
        print()

Hello World


The dataset is split into features and target, then divided into training and test sets. Hyperparameter grids are defined for models that benefit from tuning — XGBoost and Random Forest — while simpler models use default parameters. A preprocessing pipeline applies mean imputation and standard scaling to all numeric features via a ColumnTransformer, keeping the architecture extensible for mixed feature types. Each model is wrapped in a pipeline to ensure preprocessing and modeling are treated as a single unit. Models with defined parameter grids are passed through GridSearchCV, which exhaustively searches all parameter combinations using 5-fold cross-validation, selects the best combination by mean R², and exposes the best estimator for final evaluation. Models without tuning grids are evaluated using cross_val_score for an unbiased performance estimate, then refitted on the full training set for final prediction. Test R² scores are reported for all models to assess generalization performance.